# Road, Lane and Obstacle Segmentation: retraining on Google Colab

Run the cells **from top to bottom** (click a cell, press `Shift + Enter`).
If you use your own fork, change the GitHub username in step 2.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`

## 1. Check the GPU is on

In [ ]:
!nvidia-smi

## 2. Get the project code
Change the username if you're using your own fork.

In [ ]:
GITHUB_USER = "souadessemyhij1999"
!git clone https://github.com/{GITHUB_USER}/road-scene-segmentation.git
%cd road-scene-segmentation

## 3. Install libraries and download CamVid (≈2 minutes)

In [ ]:
!pip install -q -r requirements.txt
!python src/prepare_data.py

## 4. Fine-tune (≈20–30 minutes on a T4 GPU)
Starts from the YOLO26-sem model pretrained on Cityscapes (German cities) and adapts it to CamVid.

In [ ]:
!python src/train.py --epochs 60 --batch 16 --device 0

## 5. Evaluate on the 233 test images and compare with the pretrained model

In [ ]:
!python src/evaluate.py --device 0
from IPython.display import Image, display
display(Image("results/iou_comparison.png", width=750))
print(open("results/summary.md").read())

## 6. Run road, lane and obstacle analysis
Uses CamVid test frames by default. To use your own dashcam video, upload it with the icon on the left into `road-scene-segmentation/samples/` and change `SOURCE`.

In [ ]:
import glob, os, shutil
os.makedirs("samples/demo", exist_ok=True)
from ultralytics.utils import SETTINGS
for f in sorted(glob.glob(f"{SETTINGS['datasets_dir']}/camvid/images/test/Seq05VD_*.png"))[::10]:
    shutil.copy(f, "samples/demo/")
SOURCE = "samples/demo"   # e.g. "samples/my_drive.mp4"
!python src/segment.py --source {SOURCE} --out results/segmented
!python src/drive_report.py --csv results/segmented/frame_metrics.csv
display(Image("results/segmented/drive_report.png", width=900))

## 7. Download the results
This creates `results.zip` with the weights, metrics and charts.

In [ ]:
!cp results/iou_comparison.png assets/
!zip -qr results.zip weights/best.pt results/metrics_per_class.csv results/summary.md results/iou_comparison.png assets
from google.colab import files
files.download("results.zip")